# Workshop 2 — Perception

**Estimated time:** ~90 minutes  
**Prerequisites:** Workshop 1 completed; `opencv-python`, `Pillow`, `transformers`, `torch` installed

In this workshop you will:
1. Process raw camera images with OpenCV
2. Detect cube colors using HSV thresholding
3. Estimate spatial angle to objects from camera centroid
4. Run monocular depth estimation with Depth Anything v2
5. Build a `perceive()` function that returns structured scene observations

> ⚠️ **Before you begin:** Close any open simulator windows from previous workshops. The simulation has no persistence — each notebook starts a brand-new robot. Having two viewer tabs open simultaneously causes confusion.

## Section 1 — Quick Setup

This is the same connection pattern as Workshop 1. See that notebook for a detailed walkthrough.

In [ ]:
%pip install -q "menlo-robot-sdk[livekit]" python-dotenv opencv-python Pillow transformers torch matplotlib

In [ ]:
# Same setup as Workshop 1 — see that notebook for detailed explanation
import asyncio
import os
import time
import math

import cv2
import numpy as np
import PIL.Image
import matplotlib.pyplot as plt
from IPython.display import display, Image

from menlo_robot_sdk import AsyncClient, connect
from menlo_robot_sdk.experimental import generate_room_key

# ── Key loader: tries .env first, then Colab Secrets ────────────────────────
def _load_keys():
    # Mode B: local .env file
    try:
        from dotenv import load_dotenv
        load_dotenv(override=False)
        if os.environ.get("MENLO_API_KEY"):
            print("Loaded keys from .env file")
            return "dotenv"
    except ImportError:
        pass  # python-dotenv not installed → not in .env mode

    # Mode A: Google Colab Secrets
    try:
        from google.colab import userdata
        os.environ.setdefault("MENLO_API_KEY",    userdata.get("MENLO_API_KEY") or "")
        os.environ.setdefault("TOKAMAK_API_KEY",  userdata.get("TOKAMAK_API_KEY") or "")
        print("Loaded keys from Colab Secrets")
        return "colab"
    except Exception:
        pass

    return "env"  # keys already in environment (CI, Docker, etc.)

_load_keys()

MENLO_API_KEY   = os.environ.get("MENLO_API_KEY", "")
TOKAMAK_API_KEY = os.environ.get("TOKAMAK_API_KEY", "")
RCS_URL         = "https://api.menlo.ai/rcs"
VIEWER_BASE_URL = "https://sim.menlo.ai"

assert MENLO_API_KEY and not MENLO_API_KEY.startswith("sk_live_YOUR"), (
    "MENLO_API_KEY not set!\n"
    "  • Colab: add it in the Secrets manager (key icon in the left sidebar)\n"
    "  • Local: add MENLO_API_KEY=... to a .env file next to this notebook"
)
print(f"Configuration loaded — platform: {RCS_URL}")
print(f"MENLO_API_KEY    : {MENLO_API_KEY[:12]}...")
print(f"TOKAMAK_API_KEY  : {'(set)' if TOKAMAK_API_KEY else '(not set — needed for Section 7)'}")

In [ ]:
client = AsyncClient(rcs_url=RCS_URL, api_key=MENLO_API_KEY)

r = await client.robots.create(name=f"workshop2-{int(time.time())}", model="asimov-v0")
robot_id = r.robot.id
await client.robots.create_session(robot_id)

room_key = await generate_room_key(client, robot_id)
print(f"VIEWER URL:\n{VIEWER_BASE_URL}/?key={room_key}")

> Open the viewer URL in **Google Chrome**, wait for the scene to load, then run the cell below.

In [ ]:
session = await connect(
    client, robot_id,
    worker_names=[], rcw_identity_prefix="simplesim", join_livekit=True,
)
print(f"Connected: {session.session.room_name}")


async def wait_for_skills(timeout_s: float = 180.0):
    """Poll until the viewer joins and skills become available."""
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        try:
            found = await session.discover_skills()
            if found:
                return found
        except (RuntimeError, TimeoutError):
            pass  # viewer not joined yet
        await asyncio.sleep(2.0)
    raise TimeoutError("Viewer did not join — is the Chrome tab open?")


skills = await wait_for_skills()
print(f"Skills: {[s.name for s in skills]}")

**Recap:** `session.get_vision('pov')` returns raw JPEG bytes from the robot's first-person camera — introduced in Workshop 1. In this workshop we'll process those bytes with computer vision rather than just displaying them.

In [ ]:
# Helper: grab a frame as raw JPEG bytes (the perception cells below need the bytes,
# not just a display). Workshop 1's screenshot() displays the frame; this returns it.
async def grab_frame():
    return await session.get_vision("pov")

## Section 2 — Raw Image Processing with OpenCV

In [ ]:
# Grab the robot's camera
jpeg = await session.get_vision("pov")

# Decode JPEG bytes → numpy array (BGR format, as OpenCV expects)
img = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_COLOR)
print(f"Image shape: {img.shape}  (height x width x channels)")

# Display using PIL (convert BGR→RGB for correct colors)
pil_img = PIL.Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
display(pil_img)

# Crop the center region of interest (middle third)
h, w = img.shape[:2]
roi = img[h//3 : 2*h//3, w//3 : 2*w//3]
print("Center ROI:")
display(PIL.Image.fromarray(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)))

## Section 3 — Color Segmentation with HSV Thresholding

The robot's camera produces BGR images. To isolate colors robustly under different lighting, we convert to **HSV** (Hue-Saturation-Value) space and threshold each color channel.

Red wraps around the hue circle (H≈0 and H≈170), so we need two ranges combined with OR.

In [ ]:
jpeg = await session.get_vision("pov")
img = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_COLOR)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# HSV ranges for each cube color
COLOR_RANGES = {
    "red":    [(np.array([0,  50, 50]), np.array([10,  255, 255])),
               (np.array([160,50, 50]), np.array([180, 255, 255]))],
    "green":  [(np.array([40, 50, 50]), np.array([80,  255, 255]))],
    "blue":   [(np.array([100,50, 50]), np.array([130, 255, 255]))],
    "yellow": [(np.array([20, 50, 50]), np.array([35,  255, 255]))],
}

annotated = img.copy()

for color, ranges in COLOR_RANGES.items():
    # Build binary mask
    mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
    for lo, hi in ranges:
        mask |= cv2.inRange(hsv, lo, hi)

    # Find contours of blobs
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Draw bounding box for each significant blob
    BGR = {"red": (0,0,255), "green": (0,200,0), "blue": (255,0,0), "yellow": (0,200,200)}
    for cnt in contours:
        if cv2.contourArea(cnt) > 200:
            x, y, bw, bh = cv2.boundingRect(cnt)
            cv2.rectangle(annotated, (x, y), (x+bw, y+bh), BGR[color], 2)
            cv2.putText(annotated, color, (x, y-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, BGR[color], 1)

print("Annotated image with detected color blobs:")
display(PIL.Image.fromarray(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)))

## Section 4 — Centroid Estimation and Angle Offset

For navigation we need to know *where* in the frame each object is, not just whether it's there. The horizontal offset from the image center corresponds to the angle the robot needs to turn.

- **Centroid:** `(cx, cy)` = `(m10/m00, m01/m00)` using image moments
- **Angle offset:** `(cx - W/2) / (W/2) * HFOV/2` — positive = object is right of center

In [ ]:
HFOV_HALF_DEG = 30.0  # Half of horizontal field of view (~60° total for this camera)

jpeg = await session.get_vision("pov")
img = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_COLOR)
h, w = img.shape[:2]
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

print("Color detections (from camera):")
for color, ranges in COLOR_RANGES.items():
    mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
    for lo, hi in ranges:
        mask |= cv2.inRange(hsv, lo, hi)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    significant = [c for c in contours if cv2.contourArea(c) > 200]
    if not significant:
        continue
    largest = max(significant, key=cv2.contourArea)
    M = cv2.moments(largest)
    if M["m00"] == 0:
        continue
    cx = int(M["m10"] / M["m00"])
    cy = int(M["m01"] / M["m00"])
    angle_offset = (cx - w/2) / (w/2) * HFOV_HALF_DEG
    area = int(cv2.contourArea(largest))
    print(f"  {color}: centroid=({cx},{cy}), angle_offset={angle_offset:+.1f}°, area={area}px²")

# Validate: compare with scene_state ground truth
print("\nGround truth (from scene_state):")
scene = await session.state.get("scene_state")
status = await session.state.get("robot_status")
rx, ry = status.robot.pose.position[0], status.robot.pose.position[1]
yaw = status.robot.pose.yaw_deg
for eid, e in scene.entities.items():
    if eid.startswith("cube_") and e.visible:
        color = e.state.get("color", "?")
        ex, ey = e.pose.position[0], e.pose.position[1]
        bearing = math.degrees(math.atan2(ey-ry, ex-rx))
        rel_angle = (bearing - yaw + 180) % 360 - 180  # relative to robot heading
        dist = math.hypot(ex-rx, ey-ry)
        print(f"  {eid} ({color}): scene_angle={rel_angle:+.1f}°, dist={dist:.1f}m")

## Section 5 — Monocular Depth with Depth Anything v2

[Depth Anything v2](https://huggingface.co/depth-anything/Depth-Anything-V2-Small-hf) is a lightweight monocular depth estimation model. Given a single RGB image, it outputs a relative depth map.

**Note:** The first run downloads ~100 MB of model weights. Subsequent runs use the cache and are fast (~0.5s per frame on CPU).

In [ ]:
from transformers import pipeline as hf_pipeline

print("Loading Depth Anything v2 (first run downloads ~100MB)...")
depth_pipe = hf_pipeline(
    "depth-estimation",
    model="depth-anything/Depth-Anything-V2-Small-hf",
)
print("Model loaded.")

In [ ]:
jpeg = await session.get_vision("pov")
img = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_COLOR)
pil_img = PIL.Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

# Run depth estimation
result = depth_pipe(pil_img)
depth_map = np.array(result["depth"])

print(f"Depth map shape: {depth_map.shape}")
print(f"Depth range: {depth_map.min():.2f} – {depth_map.max():.2f}")

# Visualize side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.imshow(pil_img); ax1.set_title("Camera image"); ax1.axis("off")
im = ax2.imshow(depth_map, cmap="plasma")
ax2.set_title("Depth map (brighter = closer)"); ax2.axis("off")
plt.colorbar(im, ax=ax2, fraction=0.046)
plt.tight_layout()
plt.show()

# Sample depth at 3 pixel locations
h, w = depth_map.shape
samples = [(h//2, w//4, "left"), (h//2, w//2, "center"), (h//2, 3*w//4, "right")]
print("\nDepth samples:")
for row, col, label in samples:
    print(f"  {label} ({col},{row}): depth={depth_map[row, col]:.2f}")

## Section 6 — Fusing Color Detection + Depth

We can combine the two: for each color blob we detect, sample the depth map at its centroid. This gives us a rough ordering of objects by distance.

In [ ]:
jpeg = await session.get_vision("pov")
img = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_COLOR)
pil_img = PIL.Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
h, w = img.shape[:2]

depth_result = depth_pipe(pil_img)
depth_map = np.array(depth_result["depth"])

print("Color blobs with depth scores:")
detections = []
for color, ranges in COLOR_RANGES.items():
    mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
    for lo, hi in ranges:
        mask |= cv2.inRange(hsv, lo, hi)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        continue
    largest = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(largest)
    if area < 200:
        continue
    M = cv2.moments(largest)
    if M["m00"] == 0:
        continue
    cx = int(M["m10"] / M["m00"])
    cy = int(M["m01"] / M["m00"])
    # Clamp to depth map bounds
    dy = min(max(cy, 0), depth_map.shape[0]-1)
    dx = min(max(cx, 0), depth_map.shape[1]-1)
    depth_score = float(depth_map[dy, dx])
    detections.append((color, area, depth_score))

# Sort by depth score (higher = closer in Depth Anything convention)
for color, area, depth in sorted(detections, key=lambda x: -x[2]):
    print(f"  {color}: area={int(area)}px², depth_score={depth:.2f}")

## Section 7 — The `perceive()` Function

We now package everything into a single async function that returns structured observations from a camera frame.

In [ ]:
async def perceive(session):
    """Return dict of {color: {angle_deg, blob_area}} for all visible cubes."""
    jpeg = await session.get_vision("pov")
    img = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_COLOR)
    h, w = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    COLOR_RANGES_LOCAL = {
        "red":    [(np.array([0,  50, 50]), np.array([10,  255, 255])),
                   (np.array([160,50, 50]), np.array([180, 255, 255]))],
        "green":  [(np.array([40, 50, 50]), np.array([80,  255, 255]))],
        "blue":   [(np.array([100,50, 50]), np.array([130, 255, 255]))],
        "yellow": [(np.array([20, 50, 50]), np.array([35,  255, 255]))],
    }

    result = {}
    for color, ranges in COLOR_RANGES_LOCAL.items():
        mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
        for lo, hi in ranges:
            mask |= cv2.inRange(hsv, lo, hi)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        largest = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest)
        if area < 200:
            continue
        M = cv2.moments(largest)
        if M["m00"] == 0:
            continue
        cx = int(M["m10"] / M["m00"])
        angle_deg = (cx - w / 2) / (w / 2) * 30.0  # HFOV/2 = 30°
        result[color] = {"angle_deg": round(angle_deg, 1), "blob_area": int(area)}

    return result


# Demonstrate it
obs = await perceive(session)
print("perceive() output:")
for color, data in obs.items():
    print(f"  {color}: angle={data['angle_deg']:+.1f}°, area={data['blob_area']}px²")

---
## Exercises

### Exercise 1 — Annotate the Image with a Timestamp

Display the robot's POV with the current time printed in the top-left corner using `cv2.putText`.

**Hint:** `cv2.putText(img_copy, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2)`

In [ ]:
## Exercise 1: Annotate POV with timestamp
import datetime

# YOUR CODE HERE

# Expected: image displayed with white timestamp text in the top-left corner,
# e.g. "2024-06-17 14:32:01"

### Exercise 2 — Detect and Box the Green Cube

Segment the green cube using HSV thresholding (H: 40–80, S: 50–255, V: 50–255). Find the largest contour, draw a bounding box around it, and print the centroid coordinates.

In [ ]:
## Exercise 2: Detect green cube, draw bounding box, print centroid
# YOUR CODE HERE

# Expected:
# Green cube centroid: (cx=312, cy=198)
# [image with green rectangle drawn around the cube]

### Exercise 3 — Compare Camera Angle vs Scene State Angle

For each visible cube:
1. Compute the angle offset from the camera centroid (as in Section 4)
2. Compute the "ground truth" angle using `scene_state` + `robot_status` (robot yaw + atan2 to cube)
3. Print both and compute the error

How well does the camera estimate match the ground truth?

In [ ]:
## Exercise 3: Compare camera angle estimate vs scene_state ground truth
# YOUR CODE HERE

# Expected output (example):
# red:   camera=+12.3°,  scene=+10.8°,  error=1.5°
# green: camera=-5.1°,   scene=-6.2°,   error=1.1°

### Exercise 4 — Sample Depth at Detected Blob Centroids

Run Depth Anything on the current frame. For each color blob you detect, sample the depth value at its centroid. Print them sorted from closest to farthest (highest depth_score first).

In [ ]:
## Exercise 4: Depth Anything + sample at blob centroids
# YOUR CODE HERE

# Expected output (sorted closest first):
# blue:   depth_score=0.82
# red:    depth_score=0.61
# green:  depth_score=0.43

### Exercise 5 — `find_nearest_cube(jpeg_bytes)`

Implement a function `find_nearest_cube(jpeg_bytes) -> (color, angle_offset_deg)` that:
- Runs HSV segmentation on the provided JPEG bytes
- Finds the largest blob (by area) across all colors — largest blob ≈ nearest cube
- Returns `(color, angle_offset_deg)` for that blob
- Returns `(None, None)` if no blobs found

In [ ]:
## Exercise 5: find_nearest_cube
def find_nearest_cube(jpeg_bytes):
    # YOUR CODE HERE
    pass

# Test it
jpeg = await session.get_vision("pov")
color, angle = find_nearest_cube(jpeg)
print(f"Nearest cube: {color} at {angle}°")

# Expected output (example): Nearest cube: red at +12.3°
# Returns (None, None) if no cubes in view

### Exercise 6 (Stretch) — Turn Until Red Cube is Centered

Use `find_nearest_cube` (or `perceive`) in a loop:
1. Grab a frame, check if the red cube is within ±10° of center
2. If not, turn slightly: `set_velocity(wz=0.3, vx=0.1, duration_s=1.0)`
3. Stop when centered or after 12 steps

Print the angle offset each iteration.

In [ ]:
## Exercise 6 (stretch): Turn until red cube is centered
# YOUR CODE HERE

# Expected output:
# Step 1: red at +22.1° — turning right
# Step 2: red at +14.3° — turning right
# Step 3: red at +6.8°  — centered!
# (or: red not visible — turning to search)

---
## Cleanup

In [ ]:
await session.disconnect()
await client.robots.delete(robot_id)
print("Robot deleted.")